# 09 — Hyperbolic quadrant Step 1: dual-channel softmax data prep (Rating Mirror)

Prepare two visible-layer tensors for a **D-valued** (dual-channel) RBM experiment:

- **Channel 1 (`e+`)**: one-hot of the observed rating \(r\)
- **Channel 2 (`e-`)**: one-hot of the complement rating \(5.5 - r\)

**Cohort:** 51 users whose rating count is closest to the dataset median (tie-break: ascending `userId`).  
**Movie vocabulary:** top 2,000 most-rated movies in MovieLens 20M.  
**Shape:** `(n_users, n_movies, K)` with `K = 10` half-star bins.

In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

rating_path = root / "data" / "rating.csv"
summary_cache = root / "data" / "rating_stream_summary.pkl"
out_dir = root / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

assert rating_path.exists(), f"Missing {rating_path}"

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)
rating_to_idx = {float(r): i for i, r in enumerate(RATING_LEVELS)}
COMPLEMENT = {float(r): 5.5 - float(r) for r in RATING_LEVELS}

CHUNK_SIZE = 1_000_000
N_USERS_COHORT = 51
N_MOVIES_VOCAB = 2000
CSV_DTYPES = {"userId": "int32", "movieId": "int32", "rating": "float32"}

print(f"Project root: {root}")
print(f"Rating file:  {rating_path}")
print(f"Output dir:   {out_dir}")

Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
Rating file:  /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/rating.csv
Output dir:   /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed


## 1. User counts and cohort selection

In [2]:
import pickle

if summary_cache.exists():
    print(f"Loading cached stream summary: {summary_cache.name}")
    with summary_cache.open("rb") as f:
        summary = pickle.load(f)
    user_counts = summary["user_counts"]
    movie_popularity = summary["movie_popularity"]
else:
    print(f"Streaming {rating_path.name} for user/movie counts (one pass) …")
    user_counts_acc = defaultdict(int)
    movie_pop_acc = defaultdict(int)
    for chunk in pd.read_csv(rating_path, dtype=CSV_DTYPES, chunksize=CHUNK_SIZE):
        uc = chunk.groupby("userId")["movieId"].count()
        for uid, c in uc.items():
            user_counts_acc[int(uid)] += int(c)
        mc = chunk.groupby("movieId")["userId"].count()
        for mid, c in mc.items():
            movie_pop_acc[int(mid)] += int(c)
    user_counts = pd.Series(user_counts_acc, name="n_rated").sort_index()
    movie_popularity = pd.Series(movie_pop_acc, name="n_raters").sort_index()

median_rating_count = float(user_counts.median())
dist_to_median = (user_counts - median_rating_count).abs()
cohort_df = pd.DataFrame({
    "userId": user_counts.index.astype(int),
    "dist": dist_to_median.values,
}).sort_values(["dist", "userId"], ascending=[True, True]).head(N_USERS_COHORT)
cohort_user_ids = cohort_df["userId"].astype(int).tolist()
cohort_set = set(cohort_user_ids)

print(f"Median rating count (all users): {median_rating_count:.1f}")
print(f"Cohort size: {len(cohort_user_ids)}")
print(f"Cohort userIds: {cohort_user_ids}")
print(f"User 666 in cohort: {666 in cohort_set}")


Loading cached stream summary: rating_stream_summary.pkl
Median rating count (all users): 68.0
Cohort size: 51
Cohort userIds: [666, 1020, 1320, 1335, 1349, 1512, 1960, 2066, 2144, 2637, 2676, 2768, 3237, 3324, 3368, 3993, 4151, 4270, 4325, 4452, 4671, 5118, 5457, 5458, 6062, 6234, 6707, 6741, 6763, 6804, 7146, 7198, 7568, 7933, 8024, 8166, 8198, 8359, 8416, 8597, 8707, 8728, 8763, 9062, 9218, 9274, 9880, 9888, 9910, 9962, 10327]
User 666 in cohort: True


## 2. Movie vocabulary (top 2,000 by rating count)

In [3]:
movie_vocab = movie_popularity.sort_values(ascending=False).head(N_MOVIES_VOCAB).index.astype(int).tolist()
movie_to_col = {mid: j for j, mid in enumerate(movie_vocab)}
user_to_row = {uid: i for i, uid in enumerate(cohort_user_ids)}
n_users = len(cohort_user_ids)
n_movies = len(movie_vocab)

print(f"Movie vocabulary size: {n_movies}")
print(f"Most-rated movieId: {movie_vocab[0]} ({int(movie_popularity.loc[movie_vocab[0]])} ratings)")
print(f"2000th movieId: {movie_vocab[-1]} ({int(movie_popularity.loc[movie_vocab[-1]])} ratings)")

Movie vocabulary size: 2000
Most-rated movieId: 296 (67310 ratings)
2000th movieId: 94864 (2053 ratings)


## 3. Build Rating Mirror tensors

In [4]:
channel1 = np.zeros((n_users, n_movies, K), dtype=np.float32)
channel2 = np.zeros((n_users, n_movies, K), dtype=np.float32)

vocab_set = set(movie_vocab)
n_filled = 0

print("Streaming ratings for cohort × vocabulary …")
for chunk in pd.read_csv(rating_path, dtype=CSV_DTYPES, chunksize=CHUNK_SIZE):
    sub = chunk[chunk["userId"].isin(cohort_set) & chunk["movieId"].isin(vocab_set)]
    if sub.empty:
        continue
    for row in sub.itertuples(index=False):
        uid = int(row.userId)
        mid = int(row.movieId)
        r = float(row.rating)
        if r not in rating_to_idx:
            raise ValueError(f"Unexpected rating {r} for user {uid}, movie {mid}")
        rc = COMPLEMENT[r]
        if rc not in rating_to_idx:
            raise ValueError(f"Complement {rc} not in grid for rating {r}")
        i = user_to_row[uid]
        j = movie_to_col[mid]
        k1 = rating_to_idx[r]
        k2 = rating_to_idx[rc]
        channel1[i, j, k1] = 1.0
        channel2[i, j, k2] = 1.0
        n_filled += 1

print(f"Filled user-movie pairs (in vocab): {n_filled:,}")

Streaming ratings for cohort × vocabulary …
Filled user-movie pairs (in vocab): 3,152


## 4. Save arrays

In [5]:
path_c1 = out_dir / "channel1_softmax.npy"
path_c2 = out_dir / "channel2_softmax.npy"
path_users = out_dir / "cohort_user_ids.npy"
path_movies = out_dir / "movie_vocab.npy"

np.save(path_c1, channel1)
np.save(path_c2, channel2)
np.save(path_users, np.array(cohort_user_ids, dtype=np.int32))
np.save(path_movies, np.array(movie_vocab, dtype=np.int32))

print(f"Saved {path_c1}")
print(f"Saved {path_c2}")
print(f"Saved {path_users}")
print(f"Saved {path_movies}")

Saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/channel1_softmax.npy
Saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/channel2_softmax.npy
Saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/cohort_user_ids.npy
Saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/movie_vocab.npy


## 5. Verification

In [6]:
def nonzero_vector_count(arr):
    return int((arr.sum(axis=-1) > 0).sum())

print("=== Shapes ===")
print(f"channel1: {channel1.shape}")
print(f"channel2: {channel2.shape}")

nz1 = nonzero_vector_count(channel1)
nz2 = nonzero_vector_count(channel2)
print("\n=== Non-zero vectors per channel ===")
print(f"channel1: {nz1}")
print(f"channel2: {nz2}")
print(f"identical counts: {nz1 == nz2}")

# Complement symmetry: for each observed pair, k1 + k2 == 9
obs_i, obs_j = np.where(channel1.sum(axis=-1) > 0)
k1 = channel1[obs_i, obs_j, :].argmax(axis=1)
k2 = channel2[obs_i, obs_j, :].argmax(axis=1)
sym_ok = bool(np.all(k1 + k2 == 9))
print(f"\n=== Complement symmetry (k1 + k2 == 9 for all observed) ===")
print(sym_ok)
if not sym_ok:
    bad = np.where(k1 + k2 != 9)[0][:5]
    print("First mismatches:", list(zip(k1[bad], k2[bad])))

# User 666 example
assert 666 in cohort_set, "User 666 must be in cohort"
row_666 = user_to_row[666]
cols_obs = np.where(channel1[row_666].sum(axis=1) > 0)[0]
assert len(cols_obs) > 0, "User 666 has no observed movies in vocabulary"
j = int(cols_obs[0])
mid = movie_vocab[j]
k1_ex = int(channel1[row_666, j, :].argmax())
k2_ex = int(channel2[row_666, j, :].argmax())
r_orig = float(RATING_LEVELS[k1_ex])
r_comp = float(RATING_LEVELS[k2_ex])

print("\n=== User 666 example (one observed movie) ===")
print(f"movieId (vocab col {j}): {mid}")
print(f"original rating:   {r_orig}")
print(f"complement rating: {r_comp}  (5.5 - {r_orig} = {5.5 - r_orig})")
print(f"channel 1 one-hot: {channel1[row_666, j, :].astype(int).tolist()}")
print(f"channel 2 one-hot: {channel2[row_666, j, :].astype(int).tolist()}")
print(f"index sum check: {k1_ex} + {k2_ex} = {k1_ex + k2_ex}")

=== Shapes ===
channel1: (51, 2000, 10)
channel2: (51, 2000, 10)

=== Non-zero vectors per channel ===
channel1: 3152
channel2: 3152
identical counts: True

=== Complement symmetry (k1 + k2 == 9 for all observed) ===
True

=== User 666 example (one observed movie) ===
movieId (vocab col 5): 260
original rating:   4.0
complement rating: 1.5  (5.5 - 4.0 = 1.5)
channel 1 one-hot: [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
channel 2 one-hot: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
index sum check: 7 + 2 = 9
